# **Tugas & Analisis - Pertemuan III (Classification)**

**Nama Mata Kuliah:** Praktikum Penambangan Data  
**Topik:** Pertemuan III - Classification  
**Dataset:** Loan Status Prediction Dataset (`loan_data.csv`)

---
### **Nomor 1: Jelaskan apa tujuan penggunaan dataset ini?**

**Jawaban:**
Tujuan dari penggunaan dataset *Loan Status Prediction* ini adalah untuk **membangun dan mengevaluasi model klasifikasi machine learning (Supervised Learning)** yang mampu memprediksi atau menentukan apakah suatu pengajuan pinjaman (*loan application*) dari seorang nasabah/pemohon layak untuk **disetujui (`Y` - Yes)** atau **ditolak (`N` - No)** berdasarkan profil demografi, kondisi keuangan, riwayat kredit, dan karakteristik pinjaman pemohon.

Secara praktis di industri perbankan/finansial, model ini membantu mengotomatisasi proses *credit scoring* dan *credit risk assessment* sehingga mempercepat pengambilan keputusan sekaligus meminimalkan risiko kredit macet (*default risk*).

---
### **Nomor 2: Definisikan atribut yang menjadi input dan output nya?**

**Jawaban:**
1. **Input Attributes / Fitur ($X$):**
   - `Gender`: Jenis kelamin pemohon (*Male* / *Female*)
   - `Married`: Status pernikahan pemohon (*Yes* / *No*)
   - `Dependents`: Jumlah tanggungan keluarga (*0*, *1*, *2*, *3+*)
   - `Education`: Tingkat pendidikan (*Graduate* / *Not Graduate*)
   - `Self_Employed`: Status wiraswasta (*Yes* / *No*)
   - `ApplicantIncome`: Pendapatan utama pemohon (numerik)
   - `CoapplicantIncome`: Pendapatan pemohon tambahan / pasangan (numerik)
   - `LoanAmount`: Jumlah pinjaman yang diajukan (numerik, dalam ribuan)
   - `Loan_Amount_Term`: Jangka waktu pinjaman (numerik, dalam bulan)
   - `Credit_History`: Riwayat pemenuhan kredit sebelumnya (*1.0* = memenuhi kriteria, *0.0* = tidak)
   - `Property_Area`: Lokasi properti (*Urban*, *Semiurban*, *Rural*)

   *(Catatan: Kolom `Loan_ID` di-drop karena merupakan pengidentifikasi unik dan tidak memiliki nilai prediktif).*

2. **Output Attribute / Target Class ($y$):**
   - `Loan_Status`: Status persetujuan pinjaman, dengan kategori biner:
     - `Y` (Yes) = Pinjaman disetujui
     - `N` (No) = Pinjaman ditolak

---
### **Nomor 3: Silahkan membuat model klasifikasi Logistic Regression, KNN, Decision Tree, dan Random Forest untuk kasus dataset diatas!**

#### **1. Import Library**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

#### **2. Load Dataset**

In [ ]:
# Membaca dataset loan_data.csv
# Bisa menggunakan path lokal 'loan_data.csv' jika berada di folder yang sama
df = pd.read_csv('loan_data.csv')
df.head()

#### **3. Exploratory Data Analysis Singkat**

In [ ]:
print("Bentuk Dataset:", df.shape)
print("\nInformasi Dataset:")
df.info()
print("\nJumlah Missing Values:")
print(df.isnull().sum())
print("\nDistribusi Target (Loan_Status):")
print(df['Loan_Status'].value_counts())
print("\nPersentase Target:")
print((df['Loan_Status'].value_counts(normalize=True) * 100).round(2))

In [ ]:
# Visualisasi Distribusi Target
plt.figure(figsize=(6, 4))
df['Loan_Status'].value_counts().plot(kind='bar', color=['#1f77b4', '#ff7f0e'])
plt.title('Distribusi Target (Loan_Status)')
plt.xlabel('Loan Status')
plt.ylabel('Jumlah')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

#### **4. Pemisahan Fitur (X) dan Target (y), serta Data Splitting**

In [ ]:
# Drop Loan_ID karena merupakan identifier dan Loan_Status sebagai target
X = df.drop(columns=['Loan_ID', 'Loan_Status'])
y = df['Loan_Status']

# Identifikasi kolom kategorikal dan numerik
categorical_features = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']
numerical_features = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']

# Split dataset (80% Training, 20% Testing) dengan stratifikasi
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Jumlah data training: {X_train.shape[0]}")
print(f"Jumlah data testing : {X_test.shape[0]}")

#### **5. Pipeline Data Preprocessing**
- **Fitur Numerik:** Median Imputation + StandardScaler
- **Fitur Kategorikal:** Most Frequent Imputation + OneHotEncoder

In [ ]:
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, numerical_features),
    ('cat', cat_transformer, categorical_features)
])

#### **6. Inisialisasi dan Pelatihan 4 Model Klasifikasi**
1. **Logistic Regression**
2. **K-Nearest Neighbors (KNN)**
3. **Decision Tree**
4. **Random Forest**

In [ ]:
models = {
    "Logistic Regression": Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "KNN": Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', KNeighborsClassifier(n_neighbors=5))
    ]),
    "Decision Tree": Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', DecisionTreeClassifier(criterion='entropy', random_state=42))
    ]),
    "Random Forest": Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
    ])
}

# Pelatihan dan Prediksi
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions[name] = model.predict(X_test)
    print(f"Model '{name}' berhasil dilatih dan diprediksi.")

---
### **Nomor 4: Buatlah tabel yang menjelaskan performa dari model machine learning untuk kasus dataset diatas! (Kolom: Model, Accuracy, Precision, Recall)**

In [ ]:
metrics_list = []

for name, y_pred in predictions.items():
    acc = accuracy_score(y_test, y_pred)
    # Evaluasi kelas positif 'Y' (Yes)
    prec = precision_score(y_test, y_pred, pos_label='Y')
    rec = recall_score(y_test, y_pred, pos_label='Y')
    f1 = f1_score(y_test, y_pred, pos_label='Y')
    
    metrics_list.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1
    })

performance_df = pd.DataFrame(metrics_list)
# Tampilkan tabel sesuai instruksi modul praktikum
performance_df[["Model", "Accuracy", "Precision", "Recall"]]

#### **Tabel Evaluasi Lengkap (Format Tabel Markdown)**

| Model | Accuracy | Precision (pos_label='Y') | Recall (pos_label='Y') | F1-Score |
| :--- | :---: | :---: | :---: | :---: |
| **Logistic Regression** | 0.857143 (85.71%) | 0.833333 (83.33%) | 1.000000 (100.0%) | 0.909091 |
| **KNN** | 0.844156 (84.42%) | 0.841270 (84.13%) | 0.963636 (96.36%) | 0.898305 |
| **Decision Tree** | 0.792208 (79.22%) | 0.854545 (85.45%) | 0.854545 (85.45%) | 0.854545 |
| **Random Forest** | **0.883117 (88.31%)** | **0.859375 (85.94%)** | **1.000000 (100.0%)** | **0.924370** |

#### **Visualisasi Confusion Matrix untuk Masing-Masing Model**

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, (name, y_pred) in enumerate(predictions.items()):
    cm = confusion_matrix(y_test, y_pred, labels=["N", "Y"])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["N", "Y"])
    disp.plot(ax=axes[i], cmap=plt.cm.Blues, values_format='d')
    axes[i].set_title(f'Confusion Matrix: {name}')

plt.tight_layout()
plt.show()

---
### **Nomor 5: Jelaskan dari hasil eksperimen di atas, model mana yang paling baik? Jelaskan alasan anda!**

**Jawaban:**
Berdasarkan hasil eksperimen dan pengujian pada data testing, model **Random Forest** merupakan model yang **paling baik (terbaik)** di antara keempat algoritma yang diuji. 

**Alasan dan Analisis Detail:**
1. **Nilai Accuracy Tertinggi (88.31%):**
   - Random Forest memperoleh akurasi sebesar **88.31%**, mengungguli Logistic Regression (85.71%), KNN (84.42%), dan Decision Tree (79.22%).
2. **Nilai Precision Tertinggi (85.94%):**
   - Random Forest memiliki presisi tertinggi sebesar **85.94%** untuk kelas `Y`. Artinya, dari seluruh pengajuan yang diprediksi disetujui, 85.94% di antaranya benar-benar layak disetujui, sehingga meminimalkan pemberian pinjaman ke nasabah yang berisiko gagal bayar (*False Positive* rendah).
3. **Recall Sempurna (100.0%):**
   - Nilai recall kelas `Y` mencapai **1.0 (100%)**, artinya model berhasil menangkap seluruh pemohon yang memang layak disetujui pinjamannya tanpa ada yang terlewat (*False Negative* = 0).
4. **F1-Score Tertinggi (0.9244):**
   - Mengingat F1-score adalah rata-rata harmonik antara precision dan recall, Random Forest memberikan keseimbangan performa klasifikasi terbaik pada dataset ini.
5. **Keunggulan Ensemble Learning (Bagging):**
   - Random Forest menggabungkan banyak pohon keputusan (*ensemble of decision trees*) dengan *bootstrapping* dan *feature subsampling*. Hal ini mengatasi kelemahan utama single Decision Tree yang cenderung mengalami *high variance / overfitting*, sehingga menghasilkan generalisasi yang jauh lebih stabil dan akurat pada data baru.